In [1]:
import urllib.request

urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
    "input.txt"
)

('input.txt', <http.client.HTTPMessage at 0x7f4a3417b320>)

In [2]:
# read the input.txt
with open('input.txt', 'r' , encoding='utf-8') as f:
    text = f.read()

In [3]:
print("length of dataset character: ",len(text))

length of dataset character:  1115394


In [4]:
# inspect the dataset
# look at the first 1000 charcters
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [5]:
# finding vocab_size
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("vocab_size: ",vocab_size)

vocab_size:  65


In [6]:
# distinct chars / vocabulary
print(''.join(chars))


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


In [7]:
# Tokenization

# create a mapping from charcters to integers
stoi = {ch:i for i,ch in enumerate(chars)} # string to integers
iots ={i:ch for i,ch in enumerate(chars)} # integer to string

encode = lambda s:[stoi[c] for c in s] # encoder :take a string and output a list of interger
decode = lambda l:''.join([iots[i] for i in l]) # decoder : take a list of integers and output a string

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [8]:
# now encode the entire dataset and store it into tensor
import torch
data = torch.tensor(encode(text),dtype=torch.long)
print(data.shape , data.dtype)
print(data[:1000])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [9]:
# split data into train and test dataset
# taking 90% of data for the training and 10% for the testing

n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
train_data.shape , val_data.shape

(torch.Size([1003854]), torch.Size([111540]))

In [10]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [11]:
x = train_data[:block_size] # input for transformer
y = train_data[1:block_size+1]  # required output

for t in range(block_size):
  context = x[:t+1]
  target = y[t]
  print(f"when the input is {context} the target is {target} ")

when the input is tensor([18]) the target is 47 
when the input is tensor([18, 47]) the target is 56 
when the input is tensor([18, 47, 56]) the target is 57 
when the input is tensor([18, 47, 56, 57]) the target is 58 
when the input is tensor([18, 47, 56, 57, 58]) the target is 1 
when the input is tensor([18, 47, 56, 57, 58,  1]) the target is 15 
when the input is tensor([18, 47, 56, 57, 58,  1, 15]) the target is 47 
when the input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target is 58 


In [12]:
torch.manual_seed(1337)
batch_size = 4 # how many sequences pass to transformer as input
block_size = 8 # max length of the context length for prediction

def get_batch(split):
  if(split == 'train'):
    data = train_data
  else:
    data = val_data

  random_indx = torch.randint(len(data)-block_size,(batch_size,))
  x = torch.stack([data[i:i+block_size] for i in random_indx])
  y = torch.stack([data[i+1:i+block_size+1] for i in random_indx])

  return x , y

xb , yb = get_batch('train')
print("inputs: ",xb)
print("targets: ",yb)

inputs:  tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:  tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])


In [13]:
for b in range(batch_size):
  for t in range(block_size):
    context = xb[b,:t+1]
    target = yb[b,t]

    print(f" when the input is  {context.tolist()} the target is {target} ")

 when the input is  [24] the target is 43 
 when the input is  [24, 43] the target is 58 
 when the input is  [24, 43, 58] the target is 5 
 when the input is  [24, 43, 58, 5] the target is 57 
 when the input is  [24, 43, 58, 5, 57] the target is 1 
 when the input is  [24, 43, 58, 5, 57, 1] the target is 46 
 when the input is  [24, 43, 58, 5, 57, 1, 46] the target is 43 
 when the input is  [24, 43, 58, 5, 57, 1, 46, 43] the target is 39 
 when the input is  [44] the target is 53 
 when the input is  [44, 53] the target is 56 
 when the input is  [44, 53, 56] the target is 1 
 when the input is  [44, 53, 56, 1] the target is 58 
 when the input is  [44, 53, 56, 1, 58] the target is 46 
 when the input is  [44, 53, 56, 1, 58, 46] the target is 39 
 when the input is  [44, 53, 56, 1, 58, 46, 39] the target is 58 
 when the input is  [44, 53, 56, 1, 58, 46, 39, 58] the target is 1 
 when the input is  [52] the target is 58 
 when the input is  [52, 58] the target is 1 
 when the input 

In [14]:
# Bigram Language Model
import torch
import torch.nn as nn
from torch.nn import functional as f
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
  def __init__(self,vocab_size):
    super().__init__()
    # each token directly reads off the logits for the next token from a lookup table
    self.token_embedding_table = nn.Embedding(vocab_size,vocab_size)

  def forward(self,idx,targets=None):
    # idx and targets are both (batch_size , block_size(Time)) tensors of integers
    logits = self.token_embedding_table(idx) #(batch_size ,block_size(Time) , channel(vocab_size))

    if targets is None:
      loss = None
    else:
      B , T , C = logits.shape
      logits = logits.view(B*T,C)
      targets = targets.view(B*T)
      loss = f.cross_entropy(logits,targets)

    return logits , loss

  def generate(self,idx,max_new_tokens):
      # idx is (B,T) array of indices in the current context
      for _ in range(max_new_tokens):
        # get the predictions
        logits , loss = self(idx)
        #focus only on the last timestep
        logits = logits[:,-1,:] # becomes (B,C)
        # apply softmax to get the probabilities
        probs = f.softmax(logits,dim=-1) #(B,C)
        # sample from the distribution
        idx_next = torch.multinomial(probs,num_samples=1) #(B,1)
        # append sampled index to the running sequence
        idx = torch.cat((idx,idx_next),dim=1) #(B,T+1)
      return idx

model = BigramLanguageModel(vocab_size)
logits , loss = model(xb,yb)
print(logits.shape)
print(loss)

print(decode(model.generate(idx=torch.zeros((1,1),dtype=torch.long),max_new_tokens=100)[0].tolist()))

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [15]:
# train Bigram Language Model

# create a PyTorch Optimizer
optimizer = torch.optim.AdamW(model.parameters(),lr=1e-3)

batch_size = 32
for steps in range(10000):
  # sample a batch of data
  xb , yb = get_batch('train')

  # claculate loss
  logits , loss = model(xb,yb)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()

  print(loss.item())

Streaming output truncated to the last 5000 lines.
2.515348434448242
2.474428176879883
2.6085002422332764
2.6852357387542725
2.440641403198242
2.5239129066467285
2.583158016204834
2.5586931705474854
2.5336263179779053
2.5082459449768066
2.563934087753296
2.409905433654785
2.549560070037842
2.627998113632202
2.597146511077881
2.6242775917053223
2.485126495361328
2.5880191326141357
2.552927017211914
2.498333215713501
2.521122455596924
2.5981087684631348
2.56059193611145
2.47790789604187
2.60870361328125
2.561020612716675
2.542356014251709
2.6399807929992676
2.6982827186584473
2.541898250579834
2.5504093170166016
2.561579704284668
2.502772569656372
2.5175554752349854
2.681809663772583
2.603564977645874
2.5262415409088135
2.6207547187805176
2.538642168045044
2.5304956436157227
2.4993271827697754
2.671292304992676
2.6358695030212402
2.5404670238494873
2.637118339538574
2.6086719036102295
2.593581438064575
2.52583646774292
2.579291343688965
2.4875175952911377
2.552349328994751
2.628919839859

In [16]:
print(decode(model.generate(idx=torch.zeros((1,1),dtype=torch.long),max_new_tokens=500)[0].tolist()))


lso br. ave aviasurf my, yxMPZI ivee iuedrd whar ksth y h bora s be hese, woweee; the! KI 'de, ulseecherd d o blllando;LUCEO, oraingofof win!
RIfans picspeserer hee tha,
TOFonk? me ain ckntoty ded. bo'llll st ta d:
ELIS me hurf lal y, ma dus pe athouo
BEY:! Indy; by s afreanoo adicererupa anse tecorro llaus a!
OLeneerithesinthengove fal amas trr
TI ar I t, mes, n IUSt my w, fredeeyove
THek' merer, dd
We ntem lud engitheso; cer ize helorowaginte the?
Thak orblyoruldvicee chot, p,
Bealivolde Th li


In [17]:
import torch

# Determine the device where the model's parameters currently reside
model_device = next(model.parameters()).device

# Create the context tensor and explicitly move it to the model's device
context = torch.zeros((1,1),dtype=torch.long).to(model_device)

print(decode(model.generate(idx=context,max_new_tokens=500)[0].tolist()))


Fo beamen, tofr,
n s Byo tred ceathe, il ivilde w
O ff y
Fivede? ig aiMy, I ivis muofounce herevern outh f athawendesees yof th withS:

FiFLINR:

Wheader y blitow,
Ye m o ditoshyd me, ch rte u hart ararwsa
Wou fe,
INurathoune
IESSARin,
MIOLened sus;
Wh.
S:
NMy BAnind g.
iudshank
An chin is a arokisupxaseru t w ity merwo al LOLo bebte loolld worinero ya l aknge ond thal ttry b's mo ge ck.

gh, inketilllin trk$nutud t ar,
WAnt cithap's Zimponcrdistherdrtes saure ' erpoperrposthel?
Handis of hef th


In [18]:
batch_size = 64
block_size = 256
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200

n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.2

In [19]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# Masked self Attention
class Head(nn.Module):
  # head_size = n_embd // n_head
  # head_size denote the dimension of the Query,Key and Value vectors for a Single Attention Head
  def __init__(self,head_size):
    super().__init__()
    self.key = nn.Linear(n_embd,head_size,bias=False)
    self.query = nn.Linear(n_embd,head_size,bias=False)
    self.value = nn.Linear(n_embd,head_size,bias=False)

    self.register_buffer('tril',torch.tril(torch.ones(block_size,block_size)))

    self.dropout = nn.Dropout(dropout)


  def forward(self,x):
    B , T , C = x.shape  # B-->batch_size , T-->block_size(Time) , C-->n_embd(Channels)
    K = self.key(x)  # dim-->(B,T,head_size)
    Q = self.query(x) # dim-->(B,T,head_size)

    # compute similarity score(attention score)
    wt = Q @ K.transpose(-2,-1) * (K.shape[-1]**-0.5) # (B,T,head_size)@(B,T,head_size) --> (B,T,T)
    wt = wt.masked_fill(self.tril[:T,:T]==0 , float('-inf')) # dim-->(B,T,T)
    wt = F.softmax(wt,dim=-1)
    wt = self.dropout(wt)

    # weighted aggregation of values
    V = self.value(x)
    output = wt@V

    return output

In [20]:
# Masked MultiHead Attention
class MultiheadAttention(nn.Module):
  def __init__(self,n_head,head_size):
    super().__init__()
    self.heads = nn.ModuleList([Head(head_size) for _ in range(n_head)])
    self.proj =nn.Linear(n_head*head_size,n_embd)
    self.dropout = nn.Dropout(dropout)

  def forward(self,x):
    output = torch.cat([h(x) for h in self.heads],dim=-1)
    output = self.dropout(self.proj(output))
    return output

In [21]:
# Feed Forward
class FeedFoward(nn.Module):
  def __init__(self,n_embd):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(n_embd,4*n_embd),
        nn.ReLU(),
        nn.Linear(4*n_embd,n_embd),
        nn.Dropout(dropout),
    )

  def forward(self,x):
    return self.net(x)

In [22]:
# Transformer Block (Communication followed by computation)
class Block(nn.Module):
  def __init__(self,n_embd,n_head):
    super().__init__()
    head_size = n_embd // n_head
    self.sa = MultiheadAttention(n_head,head_size) # Self Attention
    self.ffwd = FeedFoward(n_embd)
    self.ln1 = nn.LayerNorm(n_embd)
    self.ln2 = nn.LayerNorm(n_embd)

  def forward(self,x):
    x = x + self.sa(self.ln1(x)) # LayerNorm before self-attention
    x = x + self.ffwd(self.ln2(x)) # LayerNorm before feedforward
    return x

In [23]:
# Mini GPT Model
class GPTLanguageModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size,n_embd)
    self.position_embedding_table = nn.Embedding(block_size,n_embd)
    self.blocks = nn.Sequential(*[Block(n_embd,n_head=n_head) for _ in range(n_layer)])
    self.ln_f = nn.LayerNorm(n_embd) # Final layer norm
    self.lm_head = nn.Linear(n_embd,vocab_size)

  def forward(self,idx,targets=None):
    B , T = idx.shape

    # idx and targets are both (B,T) tensor of integers
    tok_emb = self.token_embedding_table(idx) # (B,T,n_embd)
    pos_emb = self.position_embedding_table(torch.arange(T,device=device)) # (T,n_embd)
    x = tok_emb + pos_emb # (B,T,n_embd)
    x = self.blocks(x) # (B,T,n_embd)
    x = self.ln_f(x) # (B,T,n_embd)
    logits = self.lm_head(x) # (B,T,vocab_size)

    if targets is None:
      loss = None
    else:
      B , T , C = logits.shape
      logits = logits.view(B*T,C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits,targets)

    return logits , loss

  def generate(self,idx,max_new_tokens):
    # idx is (B,T) array of indices in the current context
    for _ in range(max_new_tokens):
      # crop idx to the last block_size tokens
      idx_cond = idx[:,-block_size:]
      # get the predictions
      logits , loss = self(idx_cond) # calls forward
      #focus only on the last timestep
      logits = logits[:,-1,:] # becomes (B,C)
      # apply softmax to get the probabilities
      probs = F.softmax(logits,dim=-1) #(B,C)
      # sample from the distribution
      idx_next = torch.multinomial(probs,num_samples=1) #(B,1)
      # append sampled index to the running sequence
      idx = torch.cat((idx,idx_next),dim=1) #(B,T+1)
    return idx

In [25]:
model = GPTLanguageModel().to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate
)

for iter in range(max_iters):

    xb, yb = get_batch('train')
    # Move input tensors to the same device as the model
    xb, yb = xb.to(device), yb.to(device)

    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if iter % eval_interval == 0:
        print(loss.item())

4.363705635070801
1.992830514907837
1.6320418119430542
1.506515622138977
1.3873209953308105
1.3254334926605225
1.263354778289795
1.2561308145523071
1.2138333320617676
1.1849310398101807
